In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

np.random.seed(42)
print("Imports successful")

In [ ]:
regions = [
    'Ashanti', 'Brong-Ahafo', 'Central', 'Eastern',
    'Greater Accra', 'Northern', 'Upper East',
    'Upper West', 'Volta', 'Western'
]

crops = ['maize', 'rice', 'cassava', 'yam', 'groundnut', 'sorghum', 'millet']

print(f"Regions: {len(regions)}")
print(f"Crops: {len(crops)}")

In [ ]:
n_samples = 1200

data = {
    'region': np.random.choice(regions, n_samples),
    'crop_type': np.random.choice(crops, n_samples),
    'planting_month': np.random.randint(1, 13, n_samples),
    'days_since_planting': np.random.randint(7, 120, n_samples),
    'soil_ph': np.round(np.random.uniform(4.5, 7.5, n_samples), 1),
    'temperature_max': np.round(np.random.uniform(28, 42, n_samples), 1),
    'temperature_min': np.round(np.random.uniform(18, 28, n_samples), 1),
    'precipitation_7day': np.round(np.random.uniform(0, 120, n_samples), 1),
    'humidity': np.round(np.random.uniform(30, 95, n_samples), 1),
}

df = pd.DataFrame(data)
print(df.shape)
df.head()

In [ ]:
def calculate_risk(row):
    risk = 0.0

    # High temperature stress
    if row['temperature_max'] > 38:
        risk += 0.25
    elif row['temperature_max'] > 35:
        risk += 0.15

    # Low rainfall = drought risk
    if row['precipitation_7day'] < 10:
        risk += 0.30
    elif row['precipitation_7day'] < 25:
        risk += 0.15

    # Excessive rainfall = flood/disease risk
    if row['precipitation_7day'] > 90:
        risk += 0.20

    # Poor soil pH
    if row['soil_ph'] < 5.0 or row['soil_ph'] > 7.0:
        risk += 0.15

    # High humidity (fungal disease risk)
    if row['humidity'] > 85:
        risk += 0.15

    # Northern regions historically more drought-prone
    if row['region'] in ['Northern', 'Upper East', 'Upper West']:
        risk += 0.10

    # Sensitive crops
    if row['crop_type'] in ['maize', 'rice']:
        risk += 0.05

    # Early growth stage is most vulnerable
    if row['days_since_planting'] < 21:
        risk += 0.10

    # Add small random noise
    risk += np.random.uniform(-0.05, 0.10)

    return 1 if risk >= 0.45 else 0

df['crop_loss_risk'] = df.apply(calculate_risk, axis=1)

print("Risk distribution:")
print(df['crop_loss_risk'].value_counts())
print(f"\nHigh risk %: {df['crop_loss_risk'].mean()*100:.1f}%")

In [ ]:
import os

output_path = '../data/crop_loss_dataset.csv'
df.to_csv(output_path, index=False)
print(f"Dataset saved: {output_path}")
print(f"Shape: {df.shape}")
df.info()